# Session 2 · Day 2 - Gap Analysis on Real Data

The sibling notebook `S2_D2_01_center_recommendation` answered one learner's question: which center
is nearest by road. Gap analysis asks the planner's question across a whole region: where are the
underserved areas, and what does "underserved" even mean. The underserved-area problem has no single
right answer, so this notebook measures it three complementary ways on the real NCR data and shows
where the three disagree.

The data is real and loads from the public bucket. The road network is built on the fly with
**OpenStreetMap** through **osmnx**, and the shortest paths come from **networkx**.

### What this notebook covers
1. Load the real NCR assessment centers, tag each by **sector**, and attach a capacity proxy
2. **Sector concentration** with the **location quotient**, mapped by city *(exercise)*
3. **Road coverage**: barangays within **5 km by road** of a center, via a reversed-graph
   multi-source Dijkstra search, rolled up per admin unit
4. **2SFCA accessibility** as the gradient alternative to the binary 5 km cutoff *(exercise)*,
   with a **Moran's I** check for clustered low access
5. Compare the **binary 5 km cutoff** against the **2SFCA gradient** and read where they disagree

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - 🌐 **Three cells reach the internet**: the two bucket reads and the OpenStreetMap download. Each
>   takes a few seconds to a couple of minutes, so read ahead while they run.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write
>   code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - ❓ **Question cells** take a short typed answer, checked against a value computed from the data.
> - 🐞 **Spot-the-flaw cells** show tempting but wrong reasoning. Read, explain, then compare to the fix.
> - 🎛️ **Play-around cells** are open to tweak: change a number, re-run, compare.
> - 🗺️ **Maps** render inline in Colab, Jupyter and VS Code. On GitHub a `folium` cell looks blank,
>   which is expected. Open the notebook in a live kernel to see it.

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install osmnx networkx geopandas shapely pyproj esda libpysal folium mapclassify

In [ ]:
import os, re, time, warnings
warnings.filterwarnings("ignore")
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg"   # let GDAL stream the remote GeoPackage

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
np.set_printoptions(precision=3, suppress=True)
ox.settings.log_console = False

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
print("osmnx", ox.__version__, "· networkx", nx.__version__, "· geopandas", gpd.__version__)

## Three ways to see a gap

"Underserved" is a modelling choice, so one number hides more than it shows.
Three lenses are built here, each catching what the previous one misses.

- **Sector concentration (location quotient).** Does a city carry its fair share of a sector's
  centers, or more, or fewer. This describes **supply mix**. Reach is a separate question.
- **Road coverage (binary 5 km cutoff).** Is a barangay within 5 km by road of a center, yes or no.
  A hard on-off line: a barangay at 4.9 km counts as covered, one at 5.1 km does not, and two
  centers 1 km away count the same as one.
- **2SFCA accessibility (a gradient).** How much center capacity reaches a barangay once competition
  for that capacity is shared out. Access falls off with distance and congestion instead of dropping
  at a boundary.

The headline comparison of this notebook is the **binary 5 km cutoff against the 2SFCA gradient**.
The binary test can call a dense region almost fully covered while the gradient still shows sharp,
clustered inequality underneath. Which picture a planner trusts changes where the next center goes.

## The centers and their sectors

Two real tables combine into the supply side.

- **`tesda_coordinates.parquet`** is the geocoded center registry from Day 1. Each row carries a
  `coord_status`, and only `valid` pins are usable for mapping and routing.
- The **2026 Accredited Competency Assessment Center** database lists, for each center, the
  **sector** and qualifications it is accredited to assess.

The two tables join on a **normalized name** (upper-cased, punctuation stripped), the linking idea
from Day 1's entity-resolution notebook. The join also yields a **capacity proxy**: the registry has
no seats-per-year column, so the number of distinct qualifications a center is accredited to assess
stands in for its throughput. One region and one sector are fixed here so all three lenses describe
the same supply.

In [ ]:
def normalize(s):
    """Upper-case, strip punctuation and collapse spaces so two spellings of one name match."""
    s = re.sub(r"[^A-Z0-9 ]+", " ", str(s).upper())
    return re.sub(r"\s+", " ", s).strip()

REGION = "NCR"                                   # admin-1 label in the registry (PSGC PH13)
SECTOR = "Automotive and Land Transportation"    # the sector all three lenses describe

centers = pd.read_parquet(BASE + "tesda_coordinates.parquet")
centers = centers[(centers["region"] == REGION) & (centers["coord_status"] == "valid")].copy()
centers["key"] = centers["name"].map(normalize)

AC_URL = BASE + "2026%20Accredited%20Compentency%20Assessment%20Center%20based%20on%20July%2015%2C%202026%20database.csv"
accred = pd.read_csv(AC_URL, dtype=str)
accred = accred[accred["Region Name"].str.contains("NCR", na=False)].copy()
accred["key"] = accred["Assessment Center"].map(normalize)

capacity = accred.groupby("key")["Qualification"].nunique().rename("capacity")   # capacity proxy
sector_sets = accred.groupby("key")["Sector Title"].apply(lambda s: set(x for x in s.dropna()))

centers = (centers[centers["key"].isin(set(accred["key"]))]
           .drop_duplicates("key")
           .merge(capacity, on="key")
           .reset_index(drop=True))
centers["in_sector"] = centers["key"].map(lambda k: SECTOR in sector_sets.get(k, set()))
print(f"{len(centers):,} {REGION} assessment centers with a valid pin and a sector tag")
print(f"{int(centers['in_sector'].sum()):,} of them assess '{SECTOR}'")
centers[["name", "city_municipality", "capacity", "in_sector"]].head()

The matched count is a **floor**. A center whose name does
not match between the two tables, or whose pin failed Day 1's coordinate check, drops out silently.
The capacity proxy is coarse too: a center accredited for ten qualifications is not literally ten
times the throughput of a one-qualification center. Both are stated assumptions, and both
belong next to any result that uses them.

## The barangays as the demand side

The barangay polygons live in `consolidated_geodata_matched.gpkg` (PSGC-coded, EPSG:4326). The file
is large, so GDAL streams only the features inside a bounding box around NCR. NCR carries the admin-1 code **`PH13`**. Each barangay's **representative point** (a point
guaranteed to fall inside the polygon) stands in for its location, and the city polygons for the map
below come from dissolving these barangays on their sub-city code.

The GeoPackage has boundaries but no population count, so demand is set
to a flat `1.0` per barangay. Every barangay then weighs the same in the accessibility step. This is a
deliberately weak proxy, and the spot-the-flaw cell near the end returns to what it hides.

In [ ]:
NCR_BBOX = (120.90, 14.30, 121.20, 14.85)                 # lon_min, lat_min, lon_max, lat_max
GPKG = "/vsicurl/" + BASE + "consolidated_geodata_matched.gpkg"

t0 = time.time()
brgys = gpd.read_file(GPKG, bbox=NCR_BBOX)
brgys = brgys[brgys["adm1_pcode"] == "PH13"].reset_index(drop=True)      # NCR only
rep = brgys.geometry.representative_point()
brgys["lon"], brgys["lat"] = rep.x.values, rep.y.values
brgys["demand"] = 1.0                                     # no population column: equal weight
print(f"{len(brgys):,} NCR barangays read in {time.time() - t0:.0f} s")

## Sector concentration with the location quotient

Raw counts mislead: a big city has the most centers of almost every sector. The fair question is
whether a sector's **share** of a city's centers is higher than its share across the whole region.
That ratio is the **location quotient (LQ)**:

$$\text{LQ}_c = \frac{\text{sector centers in } c \;/\; \text{all centers in } c}{\text{sector centers in region} \;/\; \text{all centers in region}}$$

- **LQ = 1**: the city has exactly its fair share of the sector.
- **LQ > 1**: the sector is **over-represented** there (LQ = 2 means twice the regional share).
- **LQ < 1**: under-represented.

Small denominators make the ratio jumpy. A city with three centers in total moves from LQ 0 to a
very high LQ the moment one of them happens to assess the sector. The counts must always sit next to
the ratio, or a two-center city looks like a specialisation hotspot.

### ✏️ Exercise - compute the location quotient

The grouped counts are provided in `by_city`. Fill in the three quantities that turn counts into an
LQ, and store them in the given names.

- **`regional_share`**: the sector's share across the whole region (its centers over all centers).
- **`by_city["sector_share"]`**: each city's own sector share.
- **`by_city["LQ"]`**: each city's share divided by the regional share.

*Hint:* `by_city["sector_sites"].sum()` totals a column; dividing two columns gives a new column.

In [ ]:
by_city = centers.groupby("city_municipality").agg(
    all_sites=("key", "size"), sector_sites=("in_sector", "sum"))

### START CODE HERE ###   (about 3 lines)
regional_share = None             # sector centers over all centers, region-wide
by_city["sector_share"] = None    # each city's sector share
by_city["LQ"] = None              # city share divided by the regional share
### END CODE HERE ###
by_city.sort_values("LQ", ascending=False).round(2)

## ✅ Check the result
Run this. It recomputes the LQ straight from the centers table and compares.

In [ ]:
ref = centers.groupby("city_municipality").agg(a=("key", "size"), s=("in_sector", "sum"))
rshare = ref["s"].sum() / ref["a"].sum()
ref_lq = (ref["s"] / ref["a"]) / rshare
try:
    assert regional_share is not None, "regional_share is still None. Fill in the code."
    assert by_city["LQ"].notna().all(), "the LQ column still has None. Fill in the code."
    assert np.isclose(regional_share, rshare), f"regional_share should be about {rshare:.3f}, got {regional_share}"
    assert np.allclose(by_city["LQ"].values, ref_lq.reindex(by_city.index).values), "LQ values do not match the reference"
    top = by_city["LQ"].idxmax()
    print(f"✅ Passed! Regional sector share {regional_share:.1%}. "
          f"Highest LQ: {top} ({by_city.loc[top, 'LQ']:.2f}) on only {int(by_city.loc[top, 'all_sites'])} center(s).")
except AssertionError as e:
    print("❌ Not yet:", e)

The highest LQ belongs to a city with only a handful of centers,
so one accredited center swings the ratio hard. That is the small-denominator effect. Cities with many centers give LQ values that are far more stable. The map below
colours each city by LQ, and the thin-denominator cities are the ones to read last.

In [ ]:
city_polys = brgys.dissolve(by="adm3_pcode", as_index=False)[["adm3_pcode", "geometry"]]
pts = gpd.GeoDataFrame(centers, geometry=gpd.points_from_xy(centers["longitude"], centers["latitude"]), crs="EPSG:4326")
named = gpd.sjoin(pts[["city_municipality", "geometry"]], city_polys, how="inner", predicate="within")
city_polys["city"] = city_polys["adm3_pcode"].map(
    named.groupby("adm3_pcode")["city_municipality"].agg(lambda s: s.mode().iloc[0]))
choro = city_polys.merge(by_city["LQ"], left_on="city", right_index=True, how="left")

fig, ax = plt.subplots(figsize=(8, 8))
choro.to_crs(3857).plot(ax=ax, column="LQ", cmap="RdYlGn", vmin=0, vmax=2,
                        edgecolor="white", linewidth=1.2, legend=True,
                        legend_kwds={"shrink": 0.5, "label": "LQ (1 = regional average)"},
                        missing_kwds={"color": "#dddddd"})
for _, r in choro.to_crs(3857).iterrows():
    if pd.notna(r["LQ"]):
        p = r.geometry.representative_point()
        ax.annotate(f"{str(r['city']).replace(' City', '')}\n{r['LQ']:.2f}",
                    (p.x, p.y), ha="center", fontsize=7, weight="bold")
ax.set_axis_off()
ax.set_title(f"Location quotient of '{SECTOR}'\nassessment centers, by NCR city")
plt.show()

A green city carries more than its regional share of the sector, a red city
fewer, and grey has no matched centers. The concentration map answers "where is this sector thick or
thin in the supply mix". It says nothing about whether learners can reach those centers, which is the
next lens.

## Road coverage within 5 km by road

The binary lens uses a stated rule.

> A barangay is **covered** if its representative point is within **5 km by road** of a center that
> assesses the sector.

Day 1 answered "how far is the nearest center" with one Dijkstra search per origin. NCR has thousands
of barangays, so that would be thousands of searches. Two moves reduce it to **one**.

1. **Multi-source Dijkstra.** Start the search from *all* centers at once, as if they were one big
   origin. Every road node then learns its distance to the **nearest** center, whichever it is.
2. **Reverse the graph.** Dijkstra measures distance *from* the origins. The trip of interest runs
   *from the barangay to the center* (one-way streets make the two directions differ), so every edge
   is flipped first. Searching outward from the centers on the flipped graph equals searching
   *towards* the centers on the real one.

```
real roads:      barangay --->---> center     (what a learner drives)
reversed roads:  barangay ---<---< center      (what the search walks, starting at the center)
```

`cutoff = 5000` stops the search at 5 km. Any node it never reaches is **not covered**. The search
runs on a **small, compact slice of NCR** so the OpenStreetMap download stays quick. The same method
scales to a whole region once the network is pre-built and saved.

In [ ]:
AREA_LAT, AREA_LON, AREA_M = 14.575, 121.040, 6000        # a compact, runnable slice of NCR
t0 = time.time()
G = ox.graph.graph_from_point((AREA_LAT, AREA_LON), dist=AREA_M, network_type="drive", simplify=True)
G_rev = G.reverse(copy=False)              # flip every edge: barangay->center becomes center->barangay
print(f"Drive network: {len(G):,} nodes, {G.number_of_edges():,} edges (built in {time.time() - t0:.0f} s)")

A barangay at the slice boundary may be closest to a center just outside it, so the
candidate centers include any sector center within a 2 km buffer of the slice. Each center and each barangay point is then snapped to its nearest road node.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works on single numbers or whole columns."""
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(np.subtract(lon2, lon1)) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

sec_centers = centers[centers["in_sector"]].copy()
sec_centers["d_area"] = haversine_km(AREA_LAT, AREA_LON, sec_centers["latitude"].values, sec_centers["longitude"].values)
area_centers = sec_centers[sec_centers["d_area"] <= AREA_M / 1000 + 2].copy()       # 2 km edge buffer

area_brgys = brgys[haversine_km(AREA_LAT, AREA_LON, brgys["lat"].values, brgys["lon"].values) <= AREA_M / 1000].copy()
area_centers["node"] = ox.distance.nearest_nodes(G, X=area_centers["longitude"].values, Y=area_centers["latitude"].values)
area_brgys["node"] = ox.distance.nearest_nodes(G, X=area_brgys["lon"].values, Y=area_brgys["lat"].values)
print(f"{len(area_centers)} sector centers feed the area · {len(area_brgys)} barangays to test")

In [ ]:
CUTOFF_M = 5000
t0 = time.time()
reach = nx.multi_source_dijkstra_path_length(G_rev, set(area_centers["node"]), cutoff=CUTOFF_M, weight="length")
area_brgys["road_km"] = [reach.get(n, np.nan) / 1000 for n in area_brgys["node"]]     # NaN = more than 5 km
area_brgys["covered"] = area_brgys["road_km"].notna()
print(f"One search reached {len(reach):,} road nodes within {CUTOFF_M / 1000:.0f} km ({time.time() - t0:.2f} s)")
print(f"Barangays within {CUTOFF_M / 1000:.0f} km by road of a sector center: "
      f"{int(area_brgys['covered'].sum()):,} of {len(area_brgys):,} ({area_brgys['covered'].mean():.0%})")

Rolling the barangay result up to the sub-city district (admin-3 code) gives a planner's table: the
share of each unit's barangays that clear the 5 km line. A unit near 100 percent looks solved by this lens. A unit far below it is a candidate gap.

In [ ]:
rollup = (area_brgys.groupby("adm3_pcode")["covered"]
          .agg(barangays="size", covered="sum", share_covered="mean")
          .sort_values("share_covered").round(2))
rollup

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
am = area_brgys.to_crs(3857)
am.plot(ax=ax, color=np.where(am["covered"], "#74c69d", "#f1a3a3"), edgecolor="white", linewidth=0.2)
gpd.GeoDataFrame(area_centers, geometry=gpd.points_from_xy(area_centers["longitude"], area_centers["latitude"]),
                 crs="EPSG:4326").to_crs(3857).plot(ax=ax, color="#1b4332", marker="^", markersize=45, edgecolor="white")
ax.set_axis_off()
ax.set_title(f"Green: within {CUTOFF_M / 1000:.0f} km by road of a sector center (▲)\n"
             f"{area_brgys['covered'].mean():.0%} of the slice covered")
plt.show()

In a dense region almost every barangay clears the 5 km line, so the binary map
is mostly green. By this lens the area looks close to solved. That verdict is the thing the next lens
tests, because "within 5 km of one center" says nothing about how many learners share that center or
how much of it they actually get.

## Accessibility as a gradient with 2SFCA

The **two-step floating catchment area** method replaces the on-off cutoff with a number that falls
off with distance and congestion. It works in two passes over a catchment radius.

- **Step 1 (supply ratio).** For each center *j*: `Rj = capacity_j / (demand within its catchment)`.
  A center shared by many barangays has a thinner ratio.
- **Step 2 (accessibility).** For each barangay *i*: `Ai = sum of Rj` over centers within reach.

The 5 km test gives every barangay inside the radius the
same verdict and ignores how crowded a center is. 2SFCA splits each center's capacity among the demand
that can reach it, so a barangay near one small center shared by many others scores low even though the
binary test called it covered. The gradient is the finer instrument. The cutoff is the quick screen.

Demand is the flat `1.0` per barangay set earlier, and supply is the capacity proxy. The distance
matrix is straight-line in a projected CRS (**EPSG:3123**), a stand-in for the Session 1 OSRM
travel-time matrix.

In [ ]:
def projected_xy(lon, lat):
    g = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs("EPSG:3123")
    return np.c_[g.x.values, g.y.values]

C_xy = projected_xy(sec_centers["longitude"].values, sec_centers["latitude"].values)
D_xy = projected_xy(brgys["lon"].values, brgys["lat"].values)
DM = np.sqrt(((D_xy[:, None, :] - C_xy[None, :, :]) ** 2).sum(-1))       # (barangays, centers) metres
print("distance matrix shape:", DM.shape)

### ✏️ Exercise - the 2SFCA function

Write the two passes into one reusable function for any catchment radius. Fill the body between the
markers and return the per-barangay accessibility array `Ai`.

*Hint:* `within = DM <= catchment_m` is a `(barangays, centers)` boolean mask. `within.T @ demand_pop`
sums the demand reaching each center. Guard the division where that sum is zero.

In [ ]:
def two_step_fca(DM, supply, demand_pop, catchment_m=5000):
    """Return per-barangay accessibility (2SFCA). DM is (barangays, centers) in metres."""
    ### START CODE HERE ###   (about 4 lines)
    within = None            # boolean mask: distances within the catchment
    pop_in = None            # demand served by each center
    Rj = None                # supply ratio per center (guard divide-by-zero)
    Ai = None                # accessibility per barangay
    ### END CODE HERE ###
    return Ai

## ✅ Check the result
Run this. It compares the function to an inline reference at a 5 km catchment.

In [ ]:
cap = sec_centers["capacity"].values.astype(float)
dem = brgys["demand"].values.astype(float)
within = DM <= 5000
pop_in = within.T @ dem
Rj_ref = np.divide(cap, pop_in, out=np.zeros_like(cap), where=pop_in > 0)
acc_ref = within @ Rj_ref
try:
    out = two_step_fca(DM, cap, dem, 5000)
    assert out is not None, "two_step_fca returned None, did the body get filled in?"
    assert out.shape == (DM.shape[0],), f"expected shape {(DM.shape[0],)}, got {out.shape}"
    assert np.allclose(out, acc_ref), "values do not match the reference, check the two steps"
    print("✅ Passed! The 2SFCA function matches the reference.")
except AssertionError as e:
    print("❌ Not yet:", e)

In [ ]:
acc = two_step_fca(DM, cap, dem, 5000)
brgys["access"] = acc
print(f"access: min {acc.min():.4f} · median {np.median(acc):.4f} · max {acc.max():.4f}")
print(f"{int((acc == 0).sum())} barangays have no sector center within 5 km (access 0)")

## A Moran's I check for clustered low access

A map can look patchy by chance. **Moran's I** settles whether low access clusters in space or
scatters at random. Spatial weights come from each barangay's **k = 8 nearest neighbours**
(row-standardised), and the statistic runs on the accessibility values.

Positive I with a small p-value means neighbouring barangays resemble each other, so low access forms real pockets. Clustered *low* access is exactly the gap-analysis signal.

In [ ]:
from libpysal.weights import KNN
from esda.moran import Moran

gdf = gpd.GeoDataFrame(brgys.copy(), geometry=gpd.points_from_xy(brgys["lon"], brgys["lat"]),
                       crs="EPSG:4326").to_crs("EPSG:3123")
w = KNN.from_dataframe(gdf, k=8)
w.transform = "R"
mi = Moran(acc, w)
print(f"Moran's I = {mi.I:.3f}   p (sim) = {mi.p_sim:.3f}")

`I > 0` with `p < 0.05` means accessibility is spatially **clustered**, so the
underserved barangays sit together in pockets a planner can draw a ring around. `I` near 0 would mean
no spatial pattern. The choropleth below colours each barangay by access quartile so the clusters are
visible, with the sector centers overlaid.

In [ ]:
q1, q3 = np.quantile(acc, 0.25), np.quantile(acc, 0.75)
colours = np.where(acc < q1, "#d7191c", np.where(acc > q3, "#1a9641", "#fdae61"))
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(brgys["lon"], brgys["lat"], c=colours, s=10, alpha=0.8, label="barangays")
ax.scatter(sec_centers["longitude"], sec_centers["latitude"], marker="^", c="black", s=28, label="sector centers")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Accessibility by barangay (red = least-accessible quartile)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## Binary coverage against the 2SFCA gradient

Both lenses now exist for the barangays in the road slice. The binary test called almost all of them
covered. The next cell asks a sharper question: of the barangays the 5 km test passed, how many still
land in the region's **bottom access quartile** once competition is counted. Those are the barangays a
binary map would quietly declare solved.

In [ ]:
core = area_brgys.merge(brgys[["adm4_pcode", "access"]], on="adm4_pcode", how="left")
bottom = np.quantile(brgys["access"], 0.25)
covered = core["covered"]
low_and_covered = (core["access"] < bottom) & covered
print(f"Core-slice barangays: {len(core)}")
print(f"  within 5 km by road (binary 'covered'): {covered.mean():.0%}")
print(f"  of those covered, share still in the region's bottom access quartile: "
      f"{low_and_covered.sum() / covered.sum():.0%}")
print("The binary cutoff passes them; the 2SFCA gradient flags them. That gap is the point.")

## 🎛️ Play around - the catchment radius

The catchment radius is a modelling choice. Change it and watch how many barangays
land in the bottom access quartile. A small radius starves far barangays. A large one lets one distant center rescue many. First a plain cell to edit, then a slider.

In [ ]:
catchment_km = 5          # 👈 change me and re-run (try 2, 5, 10)

acc_k = two_step_fca(DM, cap, dem, catchment_km * 1000)
n_under = int((acc_k < np.quantile(acc_k, 0.25)).sum())
n_zero = int((acc_k == 0).sum())
print(f"catchment {catchment_km} km -> {n_under} barangays in the bottom access quartile, "
      f"{n_zero} with no center in reach")

In [ ]:
# Interactive in Colab / Jupyter / VS Code (static fallback elsewhere).
try:
    from ipywidgets import interact, IntSlider
    def _show(catchment_km=5):
        a = two_step_fca(DM, cap, dem, catchment_km * 1000)
        print(f"{catchment_km} km  ->  {int((a < np.quantile(a, 0.25)).sum())} underserved, "
              f"{int((a == 0).sum())} unreachable")
    interact(_show, catchment_km=IntSlider(min=1, max=15, step=1, value=5))
except Exception:
    for km in (2, 5, 10):
        a = two_step_fca(DM, cap, dem, km * 1000)
        print(km, "km ->", int((a < np.quantile(a, 0.25)).sum()), "underserved,",
              int((a == 0).sum()), "unreachable")

### ❓ Question

Across all NCR barangays, how many fall below the **25th percentile** of accessibility, the
bottom-quartile underserved set at the 5 km catchment. Fill in the integer count.

In [ ]:
q_answer = None   # <- the answer: an integer count

In [ ]:
ref_under = int((brgys["access"] < np.quantile(brgys["access"], 0.25)).sum())
try:
    assert q_answer is not None, "q_answer is still None. Fill in the answer."
    assert int(q_answer) == ref_under, f"expected {ref_under}, got {q_answer}"
    print(f"✅ Passed! {ref_under} barangays fall in the bottom-quartile underserved set.")
except AssertionError as e:
    print("❌ Not yet:", e)

### 🐞 Spot the flaw

A teammate proposes closing the analysis like this:

```python
# Any barangay in the bottom access quartile is low demand, so no new center is needed there.
skip = brgys[brgys["access"] < np.quantile(brgys["access"], 0.25)]
```

It runs and produces a tidy list. Read the next cell, which prints what "demand" actually is in this
notebook, then explain in one sentence why calling those barangays "low demand" is wrong.

In [ ]:
low = brgys[brgys["access"] < np.quantile(brgys["access"], 0.25)]
print(f"{len(low)} barangays flagged 'low access'.")
print("demand used here:", sorted(brgys["demand"].unique()), "-> a flat 1.0 per barangay, no population file")
print("barangays with zero access (no sector center within 5 km):", int((brgys["access"] == 0).sum()))

Low **access** is a statement about supply and distance. Demand in
this notebook is a flat `1.0` per barangay because the boundary file carries no population, so a
bottom-quartile score means "few centers reach here", which is the opposite of "few people need one".
A barangay can score low precisely because it is underserved and crowded. Absence of a nearby center
is not evidence of absence of demand. The fix is to bring in a real demand measure (population, or
recorded registration and assessment counts) before any barangay is labelled low demand, and to check
that measure is not itself biased toward already-served areas.

## Recap
- Gap analysis has no single definition, so three lenses are read together. The **location quotient**
  describes the supply mix per city, the **5 km road coverage** test gives a binary reachable or not,
  and **2SFCA** turns capacity and demand into a per-barangay gradient.
- **Multi-source Dijkstra on a reversed graph** answers "how far is every barangay from the nearest
  center" in one search, and `cutoff` keeps it fast.
- In a dense region the **binary cutoff** can call almost everything covered while the **2SFCA
  gradient** still shows sharp, clustered inequality. A positive, significant **Moran's I** confirms
  the low-access pockets are real.
- Every lens rests on stated choices: the matched-name floor, the capacity proxy, the flat demand, the
  catchment radius and the 5 km threshold. Report them next to the map, and never read low access as
  low demand.

## References & further reading

Documentation and background for the tools and methods in this notebook.

- [Luo & Wang (2003), 2SFCA, Environment and Planning B, doi:10.1068/b29120](https://doi.org/10.1068/b29120)
- [Luo & Qi (2009), Enhanced 2SFCA, Health & Place, doi:10.1016/j.healthplace.2009.06.002](https://doi.org/10.1016/j.healthplace.2009.06.002)
- [Location quotient (overview)](https://en.wikipedia.org/wiki/Location_quotient)
- [OSMnx, documentation](https://osmnx.readthedocs.io/)
- [NetworkX, multi-source Dijkstra](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.multi_source_dijkstra_path_length.html)
- [GeoPandas, user guide & API](https://geopandas.org/)
- [PySAL, esda (Moran's I) and libpysal (spatial weights)](https://pysal.org/)
- [Moran's I (spatial autocorrelation)](https://en.wikipedia.org/wiki/Moran%27s_I)
- [Folium, interactive maps](https://python-visualization.github.io/folium/)
- [OpenStreetMap, about](https://www.openstreetmap.org/about)